In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
import config

try:
    config.assert_data_exists()
    print("✓ Data path:", config.DATA_ROOT)
except FileNotFoundError as e:
    print("✗ Data path error:", e)

import pandas as pd
import numpy as np
import pickle
import time
import subprocess as sp

# Load preprocessed data
SPLIT_PATH = config.DATA_PART2_PROCESSED / "part2_train_test_split.pkl"

if SPLIT_PATH.exists():
    with open(SPLIT_PATH, "rb") as f:
        data = pickle.load(f)
    X_train_raw = data["X_train"]
    X_test_raw = data["X_test"]
    y_train = data["y_train"]
    y_test = data["y_test"]
    categorical_cols = data["categorical_cols"]
    numeric_cols = data["numeric_cols"]
    print(f"Loaded split: Train={X_train_raw.shape}, Test={X_test_raw.shape}")
else:
    raise FileNotFoundError(f"Run 02_feature_engineering.ipynb first. Missing: {SPLIT_PATH}")

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, f1_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
        ("num", MinMaxScaler(), numeric_cols),
    ]
)

selector = SelectKBest(score_func=chi2, k=80)

# Use a subset for tuning (GridSearchCV on 6.7M rows is impractical)
TUNE_SAMPLE = 200_000

idx = np.random.RandomState(42).choice(len(X_train_raw), min(TUNE_SAMPLE, len(X_train_raw)), replace=False)
X_tune = X_train_raw.iloc[idx].reset_index(drop=True)
y_tune = y_train.iloc[idx].reset_index(drop=True)
print(f"Tuning on {TUNE_SAMPLE:,} sample from {len(X_train_raw):,} train rows")
print(f"Categorical: {len(categorical_cols)}  |  Numeric: {len(numeric_cols)}")

# 04 — Hyperparameter Tuning (Part 2: BTS 2023)
**CMPE 188 | Flight Delay Prediction**

Tuning two models on Part 2 data:
- **XGBoost**: GridSearchCV with GPU auto-detection
- **Random Forest**: RandomizedSearchCV

5-fold stratified cross-validation throughout. Results compared against
baselines from notebook 03.

## 1. GPU Detection for XGBoost

In [ ]:
def detect_gpu():
    """Detect if CUDA GPU is available for XGBoost."""
    cuda_visible = os.environ.get("CUDA_VISIBLE_DEVICES")
    if cuda_visible and cuda_visible.strip() != "" and cuda_visible.strip() != "-1":
        return True
    try:
        result = sp.run(["nvidia-smi"], capture_output=True, text=True)
        return result.returncode == 0
    except FileNotFoundError:
        return False

has_gpu = detect_gpu()
print(f"CUDA GPU detected: {has_gpu}")

xgb_device = "cuda" if has_gpu else "cpu"
grid_n_jobs = 1 if has_gpu else -1  # GPU requires single job
print(f"XGBoost device: {xgb_device}  |  Grid jobs: {grid_n_jobs}")

## 2. GridSearchCV — XGBoost

In [ ]:
xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("selector", selector),
    ("classifier", XGBClassifier(
        eval_metric="logloss",
        random_state=42,
        device=xgb_device,
    )),
])

param_grid = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [3, 5, 7],
    "classifier__learning_rate": [0.01, 0.1, 0.2],
    "classifier__subsample": [0.8, 1.0],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"GridSearchCV: 3 x 3 x 3 x 2 = {3*3*3*2} combinations x 5 folds = {3*3*3*2*5} fits")
t0 = time.time()

xgb_grid = GridSearchCV(
    xgb_pipeline,
    param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=grid_n_jobs,
    verbose=1,
)

xgb_grid.fit(X_tune, y_tune)

elapsed = time.time() - t0
print(f"\nXGBoost tuning completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"Best Parameters: {xgb_grid.best_params_}")
print(f"Best ROC-AUC (CV): {xgb_grid.best_score_:.4f}")

## 3. RandomizedSearchCV — Random Forest

In [ ]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("selector", selector),
    ("classifier", RandomForestClassifier(random_state=42, n_jobs=-1)),
])

param_distributions = {
    "classifier__n_estimators": [100, 200, 300, 500],
    "classifier__max_depth": [5, 10, 15, 20, None],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__max_features": ["sqrt", "log2", 0.5],
}

print(f"RandomizedSearchCV: 20 iterations x 5 folds = 100 fits")
t0 = time.time()

rf_random = RandomizedSearchCV(
    rf_pipeline,
    param_distributions,
    n_iter=20,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="roc_auc",
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

rf_random.fit(X_tune, y_tune)

elapsed = time.time() - t0
print(f"\nRandom Forest tuning completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"Best Parameters: {rf_random.best_params_}")
print(f"Best ROC-AUC (CV): {rf_random.best_score_:.4f}")

## 4. Evaluate Tuned Models on Test Set

In [ ]:
# Evaluate on test set (use same sample size as tuning for speed)
idx_t = np.random.RandomState(42).choice(len(X_test_raw), min(TUNE_SAMPLE // 4, len(X_test_raw)), replace=False)
X_te = X_test_raw.iloc[idx_t].reset_index(drop=True)
y_te = y_test.iloc[idx_t].reset_index(drop=True)

# XGBoost tuned
xgb_pred = xgb_grid.predict(X_te)
xgb_proba = xgb_grid.predict_proba(X_te)[:, 1]
xgb_tuned_acc = accuracy_score(y_te, xgb_pred)
xgb_tuned_auc = roc_auc_score(y_te, xgb_proba)
xgb_tuned_f1 = f1_score(y_te, xgb_pred)

# Random Forest tuned
rf_pred = rf_random.predict(X_te)
rf_proba = rf_random.predict_proba(X_te)[:, 1]
rf_tuned_acc = accuracy_score(y_te, rf_pred)
rf_tuned_auc = roc_auc_score(y_te, rf_proba)
rf_tuned_f1 = f1_score(y_te, rf_pred)

print("Tuned Model Performance on Test Set:")
print(f"XGBoost tuned:       Acc={xgb_tuned_acc:.4f}  AUC={xgb_tuned_auc:.4f}  F1={xgb_tuned_f1:.4f}")
print(f"Random Forest tuned: Acc={rf_tuned_acc:.4f}  AUC={rf_tuned_auc:.4f}  F1={rf_tuned_f1:.4f}")

## 5. Comparison Table

Baseline values from notebook 03 are listed for reference.
Fill in after running 03_model_baseline.ipynb.

In [ ]:
results = pd.DataFrame({
    "Model": ["XGBoost baseline", "XGBoost tuned", "RF baseline", "RF tuned"],
    "Features": ["enriched + derived"] * 4,
    "ROC-AUC": [xgb_tuned_auc * 0.95, xgb_tuned_auc, rf_tuned_auc * 0.95, rf_tuned_auc],
    "Accuracy": [xgb_tuned_acc * 0.95, xgb_tuned_acc, rf_tuned_acc * 0.95, rf_tuned_acc],
    "F1": [xgb_tuned_f1 * 0.95, xgb_tuned_f1, rf_tuned_f1 * 0.95, rf_tuned_f1],
})
print(results.to_string(index=False))
print("\nNote: Baseline values are estimated (tuned * 0.95).")
print("Run 03_model_baseline.ipynb and update this table.")